# Restore BSSs from their oldest version

Some BSSs in Google Drive have been re-saved by applications that broke their formulas, leaving hundreds of `#REF!`
errors in the Summ worksheet. This notebook replaces the current version of each affected BSS with its oldest available
Google Drive revision, converted to `.xlsx` if necessary and fully recalculated, so that the cached values in the
restored file match its formulas.

For each BSS it:

1. Finds the file from the `bss_path` metadata of the latest `original_files` materialization in Dagster.
2. Lists the Google Drive revisions of the file and downloads the oldest one that is still available.
3. Opens the revision, recalculates every formula and saves it as `.xlsx`, using Excel through
   [xlwings](https://www.xlwings.org/) if it is available, and LibreOffice otherwise.
4. If `UPLOAD` is `True`: unlocks the file in Google Drive if it is locked, uploads the recalculated workbook as a new
   version, and locks the file again. The file keeps its existing name.

Requirements:

- A development environment with access to the Dagster instance, i.e. `DAGSTER_HOME` and the Dagster database
  settings, and with `BSS_FILES_FOLDER` and `BSS_FILES_STORAGE_OPTIONS` set. The storage options must use credentials
  that can edit the BSS files.
- Excel and `pip install xlwings` on Windows or macOS, or LibreOffice. Excel's results match the values that the BSS
  authors saw. LibreOffice reports some errors differently, for example `#VALUE!` rather than `#NUM!` for the
  `=LARGE($C34:$J34,AX$8)` ranking formulas in the WB and Data worksheets.
- On macOS, Excel asks for permission to access `WORK_DIRECTORY` the first time it opens a file there.

Google Drive deletes old revisions of uploaded files after 30 days unless they are marked "Keep forever", so check the
`oldest_revision` column to see how old the restored version is. After uploading, rerun `update_external_assets` in
Dagster for the restored partitions.

In [ ]:
from pathlib import Path

# Directory for the downloaded revisions, the recalculated workbooks and LibreOffice's temporary files.
# Excel on macOS asks for permission to access each new folder, so this is a fixed location rather than a new
# temporary directory for every run.
WORK_DIRECTORY = Path.home() / "hea_restore_original"

# Restore the BSSs with more than this number of #REF! errors in the Summ worksheet of the latest original_files
# materialization. Recalculated workbooks that still have more errors than this are not uploaded.
MIN_SUMM_REF_ERRORS = 200

# Optional list of bss_files partition keys to restore instead, e.g. ["Madagascar~MG23~2017-10-31"]
PARTITION_KEYS: list[str] = []

# The recalculation engine: "excel", "libreoffice", or None to use Excel if it is available and LibreOffice otherwise
ENGINE: str | None = None

# Set to True to upload the recalculated workbooks to Google Drive. Leave it False to review the local files first.
UPLOAD = False

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from io import BytesIO

import msoffcrypto
import openpyxl
import pandas as pd
import xlrd
from dagster import AssetKey, AssetRecordsFilter, DagsterInstance
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload
from upath import UPath

try:
    import xlwings as xw
except ImportError:
    xw = None

WORK_DIRECTORY.mkdir(parents=True, exist_ok=True)

## Functions

In [ ]:
XLSX_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"

# Compound File Binary signature used by legacy .xls workbooks. Modern .xlsx workbooks are zip files.
CFB_SIGNATURE = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"

# Error values that are expected in a normal BSS, and so are not counted, matching pipelines.assets.base
IGNORED_ERROR_TYPES = ["#NUM!"]

# Usual LibreOffice locations on macOS and Windows, where the installer doesn't add it to the PATH
LIBREOFFICE_LOCATIONS = [
    "/Applications/LibreOffice.app/Contents/MacOS/soffice",
    r"C:\Program Files\LibreOffice\program\soffice.exe",
]

# LibreOffice profile settings that recalculate every formula when a workbook is loaded, instead of trusting the
# cached values saved in the file. 0 means "Always recalculate".
# EmptyStringAsZero treats "" as 0 in arithmetic. Without it, formulas such as `=IF(C89="","",C87*C88*C89)` in the
# Summ worksheet return #VALUE! in LibreOffice but 0 in Excel, because `=IF(Data!BE87="","",Data!BE87)` returns ""
# in LibreOffice when Data!BE87 refers to an empty cell, whereas Excel returns 0.
RECALCULATE_ON_LOAD_SETTINGS = """<?xml version="1.0" encoding="UTF-8"?>
<oor:items xmlns:oor="http://openoffice.org/2001/registry" xmlns:xs="http://www.w3.org/2001/XMLSchema"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
<item oor:path="/org.openoffice.Office.Calc/Formula/Load">
  <prop oor:name="OOXMLRecalcMode" oor:op="fuse"><value>0</value></prop>
</item>
<item oor:path="/org.openoffice.Office.Calc/Formula/Load">
  <prop oor:name="ODFRecalcMode" oor:op="fuse"><value>0</value></prop>
</item>
<item oor:path="/org.openoffice.Office.Calc/Formula/Syntax">
  <prop oor:name="EmptyStringAsZero" oor:op="fuse"><value>true</value></prop>
</item>
</oor:items>
"""


def get_summ_ref_errors(worksheet_errors: str) -> int:
    """
    Return the number of #REF! errors in the Summ worksheet from the worksheet_errors Markdown table.
    """
    lines = [
        [value.strip() for value in line.strip().strip("|").split("|")]
        for line in worksheet_errors.splitlines()
        if not line.startswith("|:")
    ]
    header, *rows = lines
    for row in rows:
        if row[0] == "Summ":
            return int(row[header.index("#REF!")])
    return 0


def decrypt(content: bytes) -> bytes:
    """
    Return the content of a workbook, decrypting it if Excel has encrypted it with its default password.

    Excel silently encrypts .xls files that contain locked worksheets, using the hard-coded password VelvetSweatshop.
    """
    if content[: len(CFB_SIGNATURE)] != CFB_SIGNATURE:
        return content
    office_file = msoffcrypto.OfficeFile(BytesIO(content))
    if not office_file.is_encrypted():
        return content
    office_file.load_key(password="VelvetSweatshop")
    buffer = BytesIO()
    office_file.decrypt(buffer)
    return buffer.getvalue()


def download_oldest_revision(file_id: str, directory: Path) -> tuple[Path, dict, int]:
    """
    Save the oldest available revision of a Google Drive file in `directory`.

    Return the local path, the revision metadata and the number of revisions. The local file has the extension that
    matches its contents, because Excel refuses to open files whose extension doesn't match their format.
    """
    revisions = []
    page_token = None
    while True:
        response = revisions_service.list(
            fileId=file_id,
            fields="nextPageToken,revisions(id,modifiedTime,mimeType,originalFilename,size,keepForever)",
            pageToken=page_token,
        ).execute()
        revisions.extend(response.get("revisions", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    oldest = min(revisions, key=lambda revision: revision["modifiedTime"])
    content = decrypt(revisions_service.get_media(fileId=file_id, revisionId=oldest["id"]).execute())
    extension = ".xls" if content[: len(CFB_SIGNATURE)] == CFB_SIGNATURE else ".xlsx"
    path = directory / f"oldest_revision{extension}"
    path.write_bytes(content)
    return path, oldest, len(revisions)


def count_errors(path: Path) -> Counter:
    """
    Return the number of each type of error value in each worksheet of a workbook, keyed by (worksheet, error).
    """
    errors = Counter()
    if path.suffix == ".xls":
        workbook = xlrd.open_workbook(str(path), on_demand=True)
        try:
            for sheet_name in workbook.sheet_names():
                sheet = workbook.sheet_by_name(sheet_name)
                for row in range(sheet.nrows):
                    for col in range(sheet.ncols):
                        cell = sheet.cell(row, col)
                        if cell.ctype == xlrd.XL_CELL_ERROR:
                            errors[(sheet_name, xlrd.error_text_from_code.get(cell.value, "#UNKNOWN!"))] += 1
        finally:
            workbook.release_resources()
    else:
        workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
        try:
            for worksheet in workbook.worksheets:
                for row in worksheet.iter_rows():
                    for cell in row:
                        if cell.data_type == "e":
                            errors[(worksheet.title, cell.value)] += 1
        finally:
            workbook.close()
    return Counter({key: count for key, count in errors.items() if key[1] not in IGNORED_ERROR_TYPES})


def summarize_errors(errors: Counter) -> str:
    """
    Return the total number of each type of error, e.g. "#REF!: 7, #VALUE!: 1".
    """
    totals = Counter()
    for (_, error), count in errors.items():
        totals[error] += count
    return ", ".join(f"{error}: {count}" for error, count in totals.most_common())


def excel_available() -> bool:
    """
    Return True if Excel can be controlled through xlwings.
    """
    if xw is None or sys.platform not in ("win32", "darwin"):
        return False
    try:
        if sys.platform == "darwin" and xw.apps.count:
            # Excel for macOS is already running, so don't quit it
            return True
        app = xw.App(visible=False, add_book=False)
        app.quit()
        return True
    except Exception as e:
        print(f"Excel is not available: {type(e).__name__}: {e}")
        return False


def recalculate_with_excel(source: Path, destination: Path) -> None:
    """
    Open `source` in Excel, recalculate every formula and save it to `destination` as .xlsx.
    """
    # Excel for Windows can run several instances, so use a new hidden one. Excel for macOS only runs a single
    # instance, so use it if it is already open, and leave it open afterwards.
    new_instance = sys.platform == "win32" or not xw.apps.count
    app = xw.App(visible=False, add_book=False) if new_instance else xw.apps.active
    try:
        app.display_alerts = False
        book = app.books.open(str(source), update_links=False, read_only=True)
        try:
            if sys.platform == "win32":
                app.api.CalculateFullRebuild()
            else:
                try:
                    app.api.calculate_full_rebuild()
                except AttributeError:
                    # Excel recalculates files saved by other applications or older versions of Excel when it
                    # opens them anyway, so this only affects workbooks last saved by the current version of Excel
                    print("Excel for macOS doesn't support a full rebuild; recalculating dirty cells only")
                    app.calculate()
            book.save(str(destination))
        finally:
            book.close()
    finally:
        if new_instance:
            app.quit()


def find_libreoffice() -> str:
    """
    Return the path of the LibreOffice executable.
    """
    libreoffice = (
        shutil.which("libreoffice")
        or shutil.which("soffice")
        or next((location for location in LIBREOFFICE_LOCATIONS if Path(location).is_file()), None)
    )
    if not libreoffice:
        raise RuntimeError("LibreOffice was not found. Install LibreOffice or make soffice available on PATH.")
    return libreoffice


def run_libreoffice(source: Path, output_directory: Path, profile_directory: Path) -> Path:
    """
    Convert `source` to .xlsx in `output_directory` using LibreOffice, and return the path of the converted file.

    The LibreOffice profile in `profile_directory` determines whether formulas are recalculated on load.
    """
    output_directory.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [
            find_libreoffice(),
            f"-env:UserInstallation={profile_directory.as_uri()}",
            "--headless",
            "--convert-to",
            "xlsx",
            "--outdir",
            str(output_directory),
            str(source),
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    converted = output_directory / f"{source.stem}.xlsx"
    if result.returncode or not converted.is_file():
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "LibreOffice did not create an XLSX file")
    return converted


def recalculate_with_libreoffice(source: Path, destination: Path) -> None:
    """
    Save `source` to `destination` as an .xlsx workbook with every formula recalculated by LibreOffice.

    LibreOffice trusts the cached values in .xlsx files by default, so legacy .xls files are converted to .xlsx
    first, and then the .xlsx file is loaded again using a profile that recalculates all formulas on load.
    """
    with tempfile.TemporaryDirectory(prefix="libreoffice-", dir=WORK_DIRECTORY) as temporary_directory:
        temporary_root = Path(temporary_directory)
        workbook = temporary_root / "workbook.xlsx"
        if source.suffix == ".xls":
            shutil.move(run_libreoffice(source, temporary_root / "converted", temporary_root / "profile"), workbook)
        else:
            shutil.copyfile(source, workbook)
        recalculating_profile = temporary_root / "recalculating_profile"
        (recalculating_profile / "user").mkdir(parents=True)
        (recalculating_profile / "user" / "registrymodifications.xcu").write_text(RECALCULATE_ON_LOAD_SETTINGS)
        shutil.move(run_libreoffice(workbook, temporary_root / "recalculated", recalculating_profile), destination)


def upload(file_id: str, path: Path) -> dict:
    """
    Upload `path` as a new version of a Google Drive file, unlocking the file first and locking it again afterwards.
    """
    restrictions = (
        fs.service.get(fileId=file_id, fields="contentRestrictions", supportsAllDrives=True)
        .execute()
        .get("contentRestrictions", [])
    )
    locked = any(restriction.get("readOnly") for restriction in restrictions)
    reason = next((restriction["reason"] for restriction in restrictions if restriction.get("reason")), None)
    if locked:
        fs.service.update(
            fileId=file_id, body={"contentRestrictions": [{"readOnly": False}]}, supportsAllDrives=True
        ).execute()
    try:
        response = fs.service.update(
            fileId=file_id,
            media_body=MediaIoBaseUpload(BytesIO(path.read_bytes()), mimetype=XLSX_MIME_TYPE, resumable=True),
            fields="id,version,modifiedTime",
            supportsAllDrives=True,
        ).execute()
    finally:
        # Lock the file again even if the upload failed, so that it isn't left editable
        lock = {"readOnly": True, "reason": reason or "Restored from the oldest available version"}
        fs.service.update(fileId=file_id, body={"contentRestrictions": [lock]}, supportsAllDrives=True).execute()
    return {"was_locked": locked, "uploaded_version": response.get("version"), "uploaded": response.get("modifiedTime")}

## Connect to Google Drive and choose the recalculation engine

In [ ]:
# The configured storage options are usually read only, because the pipelines never modify the BSSs
folder = UPath(
    os.environ["BSS_FILES_FOLDER"], **{**json.loads(os.environ["BSS_FILES_STORAGE_OPTIONS"]), "access": "full_control"}
)
fs = folder.fs
revisions_service = build("drive", "v3", credentials=fs.service._http.credentials, cache_discovery=False).revisions()

engine = ENGINE or ("excel" if excel_available() else "libreoffice")
recalculate = recalculate_with_excel if engine == "excel" else recalculate_with_libreoffice
print(f"Recalculating with {engine}")

## Find the BSSs to restore

Uses the `bss_path` and `worksheet_errors` metadata from the latest `original_files` materialization of each
`bss_files` partition, so materialize `original_files` first if the BSSs have changed.

In [ ]:
instance = DagsterInstance.get()
rows = []
for partition_key in PARTITION_KEYS or instance.get_dynamic_partitions("bss_files"):
    records = instance.fetch_materializations(
        AssetRecordsFilter(asset_key=AssetKey("original_files"), asset_partitions=[partition_key]), limit=1
    ).records
    if not records:
        print(f"original_files has not been materialized for {partition_key}")
        continue
    metadata = records[0].asset_materialization.metadata
    rows.append(
        {
            "partition_key": partition_key,
            "bss_path": metadata["bss_path"].value,
            "current_summ_ref_errors": get_summ_ref_errors(metadata["worksheet_errors"].value),
            "materialized": pd.Timestamp(records[0].timestamp, unit="s", tz="UTC"),
        }
    )
partitions_df = pd.DataFrame(rows)
if not PARTITION_KEYS:
    partitions_df = partitions_df[partitions_df["current_summ_ref_errors"] > MIN_SUMM_REF_ERRORS]
partitions_df = partitions_df.sort_values("partition_key", ignore_index=True)
print(f"{len(partitions_df)} BSSs to restore")
partitions_df

## Restore the BSSs

The files for each BSS are saved in a subdirectory of `WORK_DIRECTORY`. The error columns show the errors in the
oldest revision and in the recalculated workbook, so that you can check that the recalculation didn't introduce new
errors before setting `UPLOAD` to `True` and running this cell again.

In [ ]:
results = []
for row in partitions_df.itertuples():
    directory = WORK_DIRECTORY / re.sub(r"[^\w.-]+", "_", row.partition_key)
    directory.mkdir(parents=True, exist_ok=True)
    result = {"partition_key": row.partition_key, "current_summ_ref_errors": row.current_summ_ref_errors}
    try:
        file_id = fs.path_to_file_id(row.bss_path)
        revision_path, oldest, num_revisions = download_oldest_revision(file_id, directory)
        result.update(
            {
                "num_revisions": num_revisions,
                "oldest_revision": oldest["modifiedTime"],
                "oldest_filename": oldest.get("originalFilename", ""),
                "converted_from_xls": revision_path.suffix == ".xls",
            }
        )
        revision_errors = count_errors(revision_path)
        recalculated_path = directory / "recalculated.xlsx"
        recalculate(revision_path, recalculated_path)
        recalculated_errors = count_errors(recalculated_path)
        result.update(
            {
                "revision_summ_ref_errors": revision_errors[("Summ", "#REF!")],
                "recalculated_summ_ref_errors": recalculated_errors[("Summ", "#REF!")],
                "revision_errors": summarize_errors(revision_errors),
                "recalculated_errors": summarize_errors(recalculated_errors),
                "recalculated_path": str(recalculated_path),
            }
        )
        if UPLOAD:
            if recalculated_errors[("Summ", "#REF!")] > MIN_SUMM_REF_ERRORS:
                result["upload_skipped"] = "Recalculated workbook still has #REF! errors in the Summ worksheet"
            else:
                result.update(upload(file_id, recalculated_path))
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"
    print(row.partition_key, result.get("error", "OK"))
    results.append(result)

results_df = pd.DataFrame(results)
results_df